# TASK 2.1: XÂY DỰNG BỘ KIỂM THỬ SUY BIẾN VẬT LÝ (DEGRADED TEST SET)
### PIPELINE TỰ ĐỘNG SINH 6 BIẾN THỂ SUY BIẾN VẬT LÝ & ĐÓNG GÓI `test_degraded.tar` LÊN DRIVE 5TB

> **Người thực hiện**: Thành viên A *(Data & Infra Lead)*  
> **Môi trường**: Google Colab CPU đa nhân / GPU T4  
> **Hạ tầng dữ liệu**: Google Drive 5TB (`/content/drive/MyDrive/Fatformer`)  
> **Đầu ra nghiệm thu**: Tệp nén `datasets/test_degraded.tar` trên Google Drive 5TB  

---

### 1. QUY CHUẨN 6 BIẾN THỂ SUY BIẾN VẬT LÝ (DEGRADATION SPECIFICATIONS)
1. **Nén lượng tử JPEG (JPEG Compression)**:
   - `jpeg_q30`: Chất lượng $Q=30$ (Nén sâu, artifact khối $8 \times 8$ cường độ cao - Tử huyệt của CLIP)
   - `jpeg_q50`: Chất lượng $Q=50$ (Nén trung bình chuẩn chia sẻ web)
   - `jpeg_q70`: Chất lượng $Q=70$ (Nén nhẹ)
2. **Làm mờ Gauss (Gaussian Blur)**:
   - `blur_s1`: Bán kính mờ $\sigma = 1.0$
   - `blur_s2`: Bán kính mờ $\sigma = 2.0$
3. **Biến dạng mạng xã hội (Downsample-Upsample)**:
   - `down_up`: Thu nhỏ $224 \rightarrow 112$ rồi phóng to lại $224$ bằng nội suy Bicubic

---

### 2. TIÊU CHUẨN NGHIỆM THU (DEFINITION OF DONE - DoD):
- [ ] Script `tools/make_degraded.py` đa tiến trình chạy ổn định, không lỗi rò rỉ RAM.
- [ ] Bảo toàn $100\%$ cấu trúc thư mục nhị phân `0_real/` và `1_fake/` trên toàn bộ các tập con.
- [ ] Tệp `test_degraded.tar` được đóng gói an toàn và có mặt tại `/content/drive/MyDrive/Fatformer/datasets/test_degraded.tar`.

In [ ]:
# ================================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN GIT TỪ NHÁNH HOANG
# ================================================================================
from google.colab import drive
import os, sys

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Fatformer"
assert os.path.exists(DRIVE_ROOT), f"LỖI: Không tìm thấy thư mục {DRIVE_ROOT}. Vui lòng kiểm tra lại cấu trúc Drive!"
print(f"[✓] Đã kết nối thành công kho lưu trữ: {DRIVE_ROOT}")

# Đồng bộ mã nguồn Git nhánh hoang
%cd /content
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository từ branch hoang...")
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git
else:
    print("[*] Repository đã tồn tại, đang fetch và checkout nhánh hoang mới nhất...")
    %cd {REPO_DIR}
    !git fetch origin hoang
    !git checkout hoang
    !git pull origin hoang

%cd {REPO_DIR}
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")
!git log -n 1 --oneline

In [ ]:
# ================================================================================
# BƯỚC 2: GIẢI NÉN TOÀN BỘ 18 TẬP TEST CLEAN SANG Ổ SSD NVME COLAB
# ================================================================================
import os

CLEAN_DIR = "/content/dataset_local/test_clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

# 1. Giải nén 10 tập Diffusion (1.22 GB)
diff_tar = "/content/drive/MyDrive/Fatformer/datasets/test_benchmark_diffusion.tar"
if os.path.exists(diff_tar):
    print(f"[*] Đang giải nén {diff_tar} sang {CLEAN_DIR}...")
    !tar -xf "{diff_tar}" -C "{CLEAN_DIR}"
else:
    print(f"[!] CẢNH BÁO: Không tìm thấy {diff_tar}")

# 2. Giải nén 8 tập GANs (18.74 GB)
gans_tar = "/content/drive/MyDrive/Fatformer/datasets/test_benchmark_gans.tar"
if os.path.exists(gans_tar):
    print(f"[*] Đang giải nén {gans_tar} sang {CLEAN_DIR}...")
    !tar -xf "{gans_tar}" -C "{CLEAN_DIR}"
else:
    print(f"[!] CẢNH BÁO: Không tìm thấy {gans_tar}")

# Tự động dàn phẳng thư mục con diffusion_test nếu có
diff_sub = os.path.join(CLEAN_DIR, "diffusion_test")
if os.path.exists(diff_sub):
    print("[*] Đang chuẩn hóa dàn phẳng cấu trúc thư mục con diffusion_test/...")
    !cp -rn /content/dataset_local/test_clean/diffusion_test/* /content/dataset_local/test_clean/

print("\n[✓] Danh sách các thư mục test Clean hiện có:")
!ls -la /content/dataset_local/test_clean

In [ ]:
# ================================================================================
# BƯỚC 3: KIỂM TRA MÃ NGUỒN TOOLS/MAKE_DEGRADED.PY & THÔNG SỐ ĐA TIẾN TRÌNH
# ================================================================================
import multiprocessing as mp

num_cpus = mp.cpu_count()
print(f"[*] Hệ thống Colab phát hiện: {num_cpus} CPU cores.")
print("[*] Kiểm tra tham số lệnh make_degraded.py:")
!python tools/make_degraded.py --help

In [ ]:
# ================================================================================
# BƯỚC 4: CHẠY PIPELINE SINH 6 BIẾN THỂ SUY BIẾN ĐA TIẾN TRÌNH (TASK 2.1)
# ================================================================================
# Mẹo: num_workers=4 tận dụng tối đa đa nhân CPU Colab
# Hỗ trợ Resume/Skip tự động: nếu bị ngắt kết nối, chạy lại cell này sẽ không tốn thời gian làm lại ảnh cũ!

!python tools/make_degraded.py \
    --input_dir /content/dataset_local/test_clean \
    --output_root /content/dataset_local/test_degraded \
    --degradations all \
    --num_workers 4

In [ ]:
# ================================================================================
# BƯỚC 5: KIỂM ĐỊNH TÍNH TOÀN VẸN CỦA DỮ LIỆU ĐÃ SINH (AUDIT VERIFICATION)
# ================================================================================
import os
from pathlib import Path

DEG_ROOT = Path("/content/dataset_local/test_degraded")
EXPECTED_DEGS = ["jpeg_q30", "jpeg_q50", "jpeg_q70", "blur_s1", "blur_s2", "down_up"]

print("=" * 75)
print("KIỂM TOÁN TÍNH TOÀN VẸN BỘ KIỂM THỬ SUY BIẾN VẬT LÝ")
print("=" * 75)

if not DEG_ROOT.exists():
    print(f"[!] LỖI: Thư mục {DEG_ROOT} không tồn tại!")
else:
    for deg in EXPECTED_DEGS:
        deg_path = DEG_ROOT / deg
        if deg_path.exists():
            img_count = sum(1 for _ in deg_path.rglob("*.jpg"))
            print(f"• Biến thể [{deg:<10}]: {img_count:>8,d} ảnh (.jpg) -> [✓] SẴN SÀNG")
        else:
            print(f"• Biến thể [{deg:<10}]: CHƯA TỒN TẠI -> [!] THIẾU")
print("=" * 75)

In [ ]:
# ================================================================================
# BƯỚC 6: ĐÓNG GÓI TỆP TEST_DEGRADED.TAR LƯU TRỮ VỀ GOOGLE DRIVE 5TB
# ================================================================================
import os

TARGET_TAR = "/content/drive/MyDrive/Fatformer/datasets/test_degraded.tar"
print(f"[*] Đang đóng gói /content/dataset_local/test_degraded -> {TARGET_TAR}...")

!tar -cf "{TARGET_TAR}" -C /content/dataset_local test_degraded

if os.path.exists(TARGET_TAR):
    size_gb = os.path.getsize(TARGET_TAR) / (1024 ** 3)
    print(f"\n[✓] THÀNH CÔNG! Đã lưu trữ tệp nén: {TARGET_TAR}")
    print(f"• Dung lượng tệp nén: {size_gb:.2f} GB")
    print("• Sẵn sàng cho Thành viên C thực thi Task 2.3 (Benchmark Baseline Degraded)!")
else:
    print(f"[!] LỖI: Không tạo được tệp {TARGET_TAR}")